# BuildCompiler SynBioHub Authenticated Tutorial

This tutorial shows how to authenticate to SynBioHub with email/password, load inventory collections into BuildCompiler's internal SBOL document, and prepare an assembly workflow from an abstract design URI.

In [ ]:
from getpass import getpass
import sbol2

from buildcompiler.api import BuildCompiler

In [ ]:
repository_url = 'https://synbiohub.org'
email = input('SynBioHub user/email: ').strip()
password = getpass('SynBioHub password: ')

abstract_design_uri = 'https://synbiohub.org/user/Gon/abstract_design/standard_GFP/1'
collections = [
    'https://synbiohub.org/user/Gon/impl_test/impl_test_collection/1',
    'https://synbiohub.org/user/Gon/Enzyme_Implementations/Enzyme_Implementations_collection/1',
]


In [ ]:
doc = sbol2.Document()
compiler = BuildCompiler.from_synbiohub(
    collections=collections,
    repository_url=repository_url,
    email=email,
    password=password,
    sbol_doc=doc,
)

print(f'Loaded objects: {len(doc.componentDefinitions)} component definitions, {len(doc.implementations)} implementations')
print('Repository client:', type(compiler.repository_client).__name__)
print('In-memory auth token available:', compiler.repository_client.auth_token is not None)


## What BuildCompiler does with this setup

- Pulls each collection URI using authenticated `sbol2.PartShop` into `sbol_doc`.
- Reuses the same authenticated pull client for identity-based resolver misses in planning/execution.
- Uses inventory indexing to find compatible plasmids containing abstract-design parts and compatible backbones/reagents.

## Assembly simulation behavior

Legacy Golden Gate simulation now enforces strict digest validation:

- Restriction digest must yield exactly **2 fragments**.
- For part plasmids, the smaller fragment is selected as the insert.
- For the backbone plasmid, the larger fragment is selected as the backbone.
- If digest count is unexpected, simulation fails with a clear error message naming the reactant.
- Successful assembly encodes reagent usage and links generated product implementations with `wasGeneratedBy` to one assembly activity per assembled design product.